# Silver Layer — Pharmacy (`pharmacy`)

**Source:** `<source_schema>.pharmacy` (default `cdr_mimiciv.bronze.pharmacy`)  
**Target:** `<target_schema>.pharmacy` (default `cdr_mimiciv.silver.pharmacy`)

**Objective:** Standardize pharmacy medication-order records while preserving patient and admission identifiers, pharmacy and order identifiers, medication details, dosing instructions, routes, frequencies, order statuses and timestamps.

**Grain:** One source pharmacy record. The official MIMIC-IV hosp documentation states `pharmacy_id` is NOT NULL and that each row of the pharmacy table has a unique `pharmacy_id` (it links to `prescriptions` and `emar`); this notebook's profiling has not yet confirmed that uniqueness in the loaded data, so it is **profiled only (distinct count reported), not asserted**. Repeated pharmacy or order identifiers will be investigated rather than automatically removed.

**Silver rules:** Preserve every Bronze record and its clinical meaning. Standardize supported data types, validate identifiers and timestamps, inspect exact duplicates, and reconcile Bronze–Silver counts. No deletion, filtering, deduplication, imputation or aggregation.

**Data quality:** Missing values, repeated records and unusual medication timestamps will be documented and investigated. Source anomalies will not be automatically corrected.

**Source-data quality finding:** 717,515 pharmacy records
(approximately 4.02%) have `stoptime` earlier than `starttime`.
The same anomaly count was confirmed in Bronze and Silver.
The minimum reversal is 1 hour, the median is 3 hours, and the
maximum is 26,297,458 hours. All source timestamps and records
are preserved without correction.

**Publication:** The table is published with `CREATE OR REPLACE` (Delta) into `<target_schema>`; downstream Gold depends on it. Nothing is dropped. Hardcoded profiled counts are asserted only when widget `enforce_profiled_counts` is `true`.

**Cost note:** about 17.8M rows. Validation uses single-pass aggregates (no `exceptAll`, no repeated full scans).

**Dependencies:** none (reads Bronze only; no joins to other Silver tables).

In [0]:
# Import PySpark functions
from pyspark.sql import functions as F

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

SOURCE_TABLE = f"{BRONZE}.pharmacy"
TARGET_TABLE = f"{SILVER}.pharmacy"

print("Source:", SOURCE_TABLE)
print("Target:", TARGET_TABLE)
print("Enforce profiled counts:", ENFORCE_PROFILED_COUNTS)

Source: cdr_mimiciv.bronze.pharmacy
Target: cdr_mimiciv.silver.pharmacy
Enforce profiled counts: True


In [0]:
# Load and inspect pharmacy medication-order records
bronze_df = spark.table(SOURCE_TABLE)

bronze_df.printSchema()
display(bronze_df.limit(20))

root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- pharmacy_id: integer (nullable = true)
 |-- poe_id: string (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- medication: string (nullable = true)
 |-- proc_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- entertime: timestamp (nullable = true)
 |-- verifiedtime: timestamp (nullable = true)
 |-- route: string (nullable = true)
 |-- frequency: string (nullable = true)
 |-- disp_sched: string (nullable = true)
 |-- infusion_type: string (nullable = true)
 |-- sliding_scale: string (nullable = true)
 |-- lockout_interval: string (nullable = true)
 |-- basal_rate: double (nullable = true)
 |-- one_hr_max: string (nullable = true)
 |-- doses_per_24_hrs: integer (nullable = true)
 |-- duration: double (nullable = true)
 |-- duration_interval: string (nullable = true)
 |-- expiration_value: integer (nullable = true)
 |

subject_id,hadm_id,pharmacy_id,poe_id,starttime,stoptime,medication,proc_type,status,entertime,verifiedtime,route,frequency,disp_sched,infusion_type,sliding_scale,lockout_interval,basal_rate,one_hr_max,doses_per_24_hrs,duration,duration_interval,expiration_value,expiration_unit,expirationdate,dispensation,fill_quantity
10000032,22595853,12775705,10000032-55,2180-05-08T08:00:00.000Z,2180-05-07T22:00:00.000Z,Furosemide,Unit Dose,Discontinued via patient discharge,2180-05-07T09:32:35.000Z,2180-05-07T09:32:35.000Z,PO/NG,DAILY,08,null,null,null,null,null,1,null,Ongoing,36,Hours,null,Omnicell,null
10000032,22595853,18415984,10000032-42,2180-05-07T02:00:00.000Z,2180-05-07T22:00:00.000Z,Ipratropium Bromide Neb,Unit Dose,Discontinued via patient discharge,2180-05-07T01:49:23.000Z,2180-05-07T01:49:23.000Z,IH,Q6H,"2, 8, 14, 20",null,null,null,null,null,4,null,Ongoing,36,Hours,null,Omnicell,null
10000032,22595853,23637373,10000032-35,2180-05-07T01:00:00.000Z,2180-05-07T09:00:00.000Z,Furosemide,Unit Dose,Inactive (Due to a change order),2180-05-07T00:09:24.000Z,2180-05-07T00:09:24.000Z,PO/NG,DAILY,08,null,null,null,null,null,1,null,Ongoing,36,Hours,null,Omnicell,null
10000032,22595853,26862314,10000032-41,2180-05-07T01:00:00.000Z,2180-05-07T01:00:00.000Z,Potassium Chloride,Unit Dose,Discontinued,2180-05-07T00:09:24.000Z,2180-05-07T00:09:24.000Z,PO,ONCE,1,null,null,null,null,null,1,1.0,Doses,36,Hours,null,Omnicell,null
10000032,22595853,30740602,10000032-27,2180-05-07T00:00:00.000Z,2180-05-07T22:00:00.000Z,Sodium Chloride 0.9% Flush,Unit Dose,Discontinued via patient discharge,2180-05-07T00:00:54.000Z,2180-05-07T00:00:54.000Z,IV,Q8H,"0, 8, 16",null,null,null,null,null,3,null,Ongoing,36,Hours,null,Floor Stock Item,null
10000032,22595853,43092008,10000032-37,2180-05-07T01:00:00.000Z,2180-05-07T22:00:00.000Z,Raltegravir,Unit Dose,Discontinued via patient discharge,2180-05-07T00:09:24.000Z,2180-05-07T00:09:24.000Z,PO,BID,"08, 20",null,null,null,null,null,2,null,Ongoing,36,Hours,null,Omnicell,null
10000032,22595853,46308055,10000032-56,2180-05-08T08:00:00.000Z,2180-05-07T22:00:00.000Z,Spironolactone,Unit Dose,Discontinued via patient discharge,2180-05-07T09:32:35.000Z,2180-05-07T09:32:35.000Z,PO/NG,DAILY,08,null,null,null,null,null,1,null,Ongoing,36,Hours,null,Omnicell,null
10000032,22595853,48214508,10000032-39,2180-05-07T01:00:00.000Z,2180-05-07T22:00:00.000Z,Acetaminophen,Unit Dose,Discontinued via patient discharge,2180-05-07T00:09:24.000Z,2180-05-07T00:09:24.000Z,PO/NG,Q6H:PRN,null,null,null,null,null,null,null,null,Ongoing,36,Hours,null,Omnicell,null
10000032,22595853,71489491,10000032-23,2180-05-07T00:00:00.000Z,2180-05-07T22:00:00.000Z,Influenza Vaccine Quadrivalent,Unit Dose,Discontinued via patient discharge,2180-05-07T00:00:54.000Z,2180-05-07T00:00:54.000Z,IM,NOW X1,null,null,null,null,null,null,0,null,Ongoing,36,Hours,null,Omnicell,null
10000032,22595853,73243129,10000032-33,2180-05-07T01:00:00.000Z,2180-05-07T22:00:00.000Z,Albuterol Inhaler,Unit Dose,Discontinued via patient discharge,2180-05-07T00:09:24.000Z,2180-05-07T00:09:24.000Z,IH,Q4H:PRN,null,null,null,null,null,null,null,null,Ongoing,36,Hours,null,Bulk Item,null


In [0]:
%sql
Select distinct route
from cdr_mimiciv.bronze.pharmacy;



route
PO
IC
TT
IM
IP
null
SUBCUT
IVPCA
IJ
SC


In [0]:
# Preserve all columns and standardize supported identifiers and timestamps
silver_df = bronze_df

long_columns = [
    "subject_id", "hadm_id", "pharmacy_id",
    "poe_seq"
]

timestamp_columns = [
    "starttime", "stoptime"
]

for c in long_columns:
    if c in silver_df.columns:
        silver_df = silver_df.withColumn(
            c, F.col(c).cast("long")
        )

for c in timestamp_columns:
    if c in silver_df.columns:
        silver_df = silver_df.withColumn(
            c, F.col(c).cast("timestamp")
        )

silver_df.printSchema()

# Ordered expected schema (Bronze columns, with the casts above applied)
EXPECTED_SCHEMA = {name: dtype for name, dtype in silver_df.dtypes}
for c in long_columns:
    if c in EXPECTED_SCHEMA:
        assert EXPECTED_SCHEMA[c] == "bigint", f"{c} not bigint"
for c in timestamp_columns:
    if c in EXPECTED_SCHEMA:
        assert EXPECTED_SCHEMA[c] == "timestamp", f"{c} not timestamp"

root
 |-- subject_id: long (nullable = true)
 |-- hadm_id: long (nullable = true)
 |-- pharmacy_id: long (nullable = true)
 |-- poe_id: string (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- medication: string (nullable = true)
 |-- proc_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- entertime: timestamp (nullable = true)
 |-- verifiedtime: timestamp (nullable = true)
 |-- route: string (nullable = true)
 |-- frequency: string (nullable = true)
 |-- disp_sched: string (nullable = true)
 |-- infusion_type: string (nullable = true)
 |-- sliding_scale: string (nullable = true)
 |-- lockout_interval: string (nullable = true)
 |-- basal_rate: double (nullable = true)
 |-- one_hr_max: string (nullable = true)
 |-- doses_per_24_hrs: integer (nullable = true)
 |-- duration: double (nullable = true)
 |-- duration_interval: string (nullable = true)
 |-- expiration_value: integer (nullable = true)
 |-- expira

In [0]:
# Single pass over Bronze: row count, unsafe casts and reversed ranges
cast_checks = {
    "subject_id": "bigint",
    "hadm_id": "bigint",
    "pharmacy_id": "bigint",
    "poe_seq": "bigint",
    "starttime": "timestamp",
    "stoptime": "timestamp"
}
available_cast_checks = {
    c: dtype for c, dtype in cast_checks.items() if c in bronze_df.columns
}

bronze_aggs = [F.count("*").alias("rows")]
bronze_aggs += [
    F.sum(
        F.when(
            F.col(c).isNotNull()
            & F.expr(f"try_cast(`{c}` AS {dtype})").isNull(),
            1
        ).otherwise(0)
    ).alias(f"invalid_cast_{c}")
    for c, dtype in available_cast_checks.items()
]
if {"starttime", "stoptime"}.issubset(bronze_df.columns):
    bronze_aggs.append(
        F.sum(
            (
                F.expr("try_cast(starttime AS timestamp)").isNotNull()
                & F.expr("try_cast(stoptime AS timestamp)").isNotNull()
                & (
                    F.expr("try_cast(stoptime AS timestamp)")
                    < F.expr("try_cast(starttime AS timestamp)")
                )
            ).cast("long")
        ).alias("reversed_times")
    )

bronze_stats = bronze_df.agg(*bronze_aggs).first().asDict()
display(spark.createDataFrame([(k, int(v or 0)) for k, v in bronze_stats.items()], ["metric", "value"]))

bronze_count = bronze_stats["rows"]
bronze_invalid_times = int(bronze_stats.get("reversed_times") or 0)
cast_issues = {k: v for k, v in bronze_stats.items() if k.startswith("invalid_cast_") and v}

print("Bronze rows:", bronze_count)
print("Bronze reversed timestamps:", bronze_invalid_times)

metric,value
rows,17847567
invalid_cast_subject_id,0
invalid_cast_hadm_id,0
invalid_cast_pharmacy_id,0
invalid_cast_starttime,0
invalid_cast_stoptime,0
reversed_times,717515


Bronze rows: 17847567
Bronze reversed timestamps: 717515


In [0]:
# Single pass over Silver: row count, missingness and reversed ranges
critical_columns = [
    "subject_id", "hadm_id", "pharmacy_id",
    "poe_id", "starttime", "stoptime", "medication"
]
available_critical_columns = [
    c for c in critical_columns if c in silver_df.columns
]

has_times = {"starttime", "stoptime"}.issubset(silver_df.columns)
reversed_cond = (
    F.col("starttime").isNotNull()
    & F.col("stoptime").isNotNull()
    & (F.col("stoptime") < F.col("starttime"))
) if has_times else F.lit(False)
difference_hours = (
    (F.unix_timestamp("starttime") - F.unix_timestamp("stoptime")) / 3600.0
) if has_times else F.lit(None).cast("double")

silver_aggs = [F.count("*").alias("rows")]
silver_aggs += [
    F.sum(F.col(c).isNull().cast("long")).alias(f"null_{c}")
    for c in available_critical_columns
]
if "pharmacy_id" in silver_df.columns:
    # Profiled only: uniqueness of pharmacy_id is NOT asserted
    silver_aggs.append(F.countDistinct("pharmacy_id").alias("distinct_pharmacy_id"))
silver_aggs += [
    F.sum(reversed_cond.cast("long")).alias("reversed_times"),
    F.min(F.when(reversed_cond, difference_hours)).alias("reversed_min_hours"),
    F.percentile_approx(F.when(reversed_cond, difference_hours), 0.5).alias("reversed_median_hours"),
    F.max(F.when(reversed_cond, difference_hours)).alias("reversed_max_hours"),
]

silver_stats = silver_df.agg(*silver_aggs).first().asDict()
display(spark.createDataFrame([(k, None if v is None else float(v)) for k, v in silver_stats.items()], ["metric", "value"]))

silver_count = silver_stats["rows"]
silver_invalid_times = int(silver_stats["reversed_times"] or 0)

print("Silver rows:", silver_count)
print("Silver reversed timestamps (preserved, review only):", silver_invalid_times)

metric,value
rows,1.7847567E7
null_subject_id,0.0
null_hadm_id,0.0
null_pharmacy_id,0.0
null_poe_id,145597.0
null_starttime,21894.0
null_stoptime,92169.0
null_medication,1137574.0
distinct_pharmacy_id,1.7847567E7
reversed_times,717515.0


Silver rows: 17847567
Silver reversed timestamps (preserved, review only): 717515


In [0]:
# Inspect exact duplicate pharmacy records without removing them (single pass)
dup_row = (
    silver_df
    .groupBy(*silver_df.columns)
    .count()
    .filter(F.col("count") > 1)
    .agg(
        F.count("*").alias("duplicate_groups"),
        F.coalesce(F.sum("count"), F.lit(0)).alias("records_in_duplicate_groups"),
        F.coalesce(F.max("count"), F.lit(0)).alias("maximum_group_size")
    )
    .first()
)
duplicate_count = dup_row["duplicate_groups"]

print("Exact duplicate pharmacy groups:", duplicate_count)
print("Records in duplicate groups:", dup_row["records_in_duplicate_groups"])
print("Maximum group size:", dup_row["maximum_group_size"])

Exact duplicate pharmacy groups: 0
Records in duplicate groups: 0
Maximum group size: 0


In [0]:
# Pre-publish validation (row preservation, safe casts, required identifiers)
required_columns = ["subject_id", "hadm_id"]
issues = {
    f"null_{c}": silver_stats[f"null_{c}"]
    for c in required_columns
    if f"null_{c}" in silver_stats and silver_stats[f"null_{c}"] != 0
}

assert bronze_count == silver_count, "Bronze–Silver row-count mismatch"
assert not issues, f"Critical pharmacy quality issues: {issues}"
assert not cast_issues, f"Unsafe type conversions: {cast_issues}"
assert bronze_invalid_times == silver_invalid_times, (
    "Timestamp anomaly counts differ between Bronze and Silver"
)

if ENFORCE_PROFILED_COUNTS:
    assert duplicate_count == 0, "Exact duplicate pharmacy groups require review"
    assert silver_invalid_times == 717515, (
        "Timestamp anomaly count changed; review before publication"
    )

print("All pre-publish checks passed.")
print("Invalid timestamp ranges (review only):", silver_invalid_times)

All pre-publish checks passed.
Invalid timestamp ranges (review only): 717515


In [0]:
spark.sql("DROP TABLE IF EXISTS cdr_mimiciv.silver.pharmacy")

DataFrame[]

In [0]:
# Publish all validated records (CREATE OR REPLACE), including documented source timestamp anomalies
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

silver_df.writeTo(TARGET_TABLE).using("delta").createOrReplace()

print("Published:", TARGET_TABLE)
print("Preserved source timestamp anomalies:", silver_invalid_times)

Published: cdr_mimiciv.silver.pharmacy
Preserved source timestamp anomalies: 717515


In [0]:
# Verify the published Silver pharmacy table (single aggregate pass)
published_df = spark.table(TARGET_TABLE)

published_schema = {name: dtype for name, dtype in published_df.dtypes}
assert list(published_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Published schema differs from expected: {published_schema}"
)

published_stats = published_df.agg(
    F.count("*").alias("rows"),
    *[
        F.sum(F.col(c).isNull().cast("long")).alias(f"null_{c}")
        for c in required_columns if c in published_df.columns
    ]
).first().asDict()

print("Bronze rows:", bronze_count)
print("Published rows:", published_stats["rows"])

assert published_stats["rows"] == bronze_count, "Published row-count mismatch"
assert all(v == 0 for k, v in published_stats.items() if k.startswith("null_")), (
    "Published required identifiers contain NULLs"
)

print("Silver pharmacy table verified successfully.")

Bronze rows: 17847567
Published rows: 17847567
Silver pharmacy table verified successfully.


In [0]:
dbutils.notebook.exit("pharmacy:OK")